In [3]:
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

# ========================
# 1. Load data
# ========================

# Query diambil dari Excel
query_dataframe = pd.read_excel('step6_final.xlsx')  # Ganti nama file jika perlu
query_data = query_dataframe['hasil_preprocessing'].dropna().tolist()  # Pastikan tidak ada NaN

# Korpus adalah daftar isu pemilu (dimasukkan sebagai list Python)
corpus_data = [
          'Adanya kekerasan/kerusuhan berbasis SARA dalam isu-isu politik dan kepemiluan.',
          'Adanya kekerasan/kerusuhan yang melibatkan tokoh publik/politik/aparat keamanan.',
          'Adanya bencana alam yang mengganggu tahapan Pemilu/Pilkada.',
          'Adanya bencana non alam (pandemi covid 19) yang mengganggu tahapan Pemilu/Pilkada.',
          'Adanya Intimidasi Terhadap Peserta Pemilu dalam proses pelaksanaan Pemilu/Pilkada.',
          'Adanya Intimidasi Terhadap Penyelenggara Pemilu dalam proses pelaksanaan Pemilu/Pilkada.',
          'Adanya Intimidasi Terhadap Pemilih dalam proses pelaksanaan Pemilu/Pilkada.',
          'Adanya perusakan fasilitas penyelenggaraan Pemilu.',
          'Adanya putusan DKPP yang ditujukan kepada Jajaran KPU dan/atau Bawaslu.',
          'Adanya Rekomendasi dan/atau Putusan Bawaslu yang tidak ditindaklanjuti oleh KPU.',
          'Adanya rekomendasi Bawaslu terkait dengan perubahan suara pada proses rekapitulasi suara.',
          'Adanya Rekomendasi Bawaslu terkait ketidaknetralan ASN/TNI/POLRI.',
          'Adanya penyalahgunaan anggaran Pemilu/Pilkada.',
          'Adanya intimidasi kepada penyelenggara pemilu yang dilakukan oleh Kepala Daerah.',
          'Adanya himbauan dan/atau tindakan untuk memilih calon tertentu dari pemerintah lokal.',
          'Adanya himbauan dan/atau tindakan untuk menolak calon tertentu dari pemerintah lokal atau tokoh masyarakat.',
          'Adanya pemilih memenuhi syarat tetapi tidak terdaftar dalam daftar pemilih tetap.',
          'Adanya pemilih tidak memenuhi syarat terdaftar dalam pemilih tetap.',
          'Adanya penduduk potensial memilih tetapi tidak memiliki KTP-Elektronik.',
          'Adanya pemilih ganda dalam daftar pemilih (logistik).',
          'Adanya penggelembungan jumlah data pemilih.',
          'Adanya Pemilih Pindah Memilih (DPTB) yang Tidak Dapat Memberikan Hak Suaranya.',
          'Adanya ketidakprofesionalan Penyelenggara Pemilu yang merugikan kampanye calon.',
          'Adanya kampanye yang difasilitasi KPU tidak sesuai dengan ketentuan.',
          'Adanya penyelenggara Pemilu yang menunjukan sikap keberpihakan dalam tahapan kampanye.',
          'Adanya penyelenggara Pemilu yang melakukan pengabaian terhadap protokol kesehatan dalam kegiatan kampanye.',
          'Adanya perlengkapan pemungutan suara yang tidak sesuai dengan ketentuan Ada keterlambatan perlengkapan (logistik) pemungutan suara.',
          'Ada keterlambatan perlengkapan (logistik) pemungutan suara.',
          'Adanya logistik berupa surat suara pemungutan suara yang tertukar.',
          'Adanya informasi tidak terdapat TPS untuk menfasilitasi pemungutan suara bagi pemilih di LAPAS/RS/SEKOLAH DINAS/.',
          'Adanya informasi pelanggaraan saat pemungutan suara di Pemilu/Pilkada.',
          'Adanya pemungutan suara ulang di Pemilu/Pilkada.',
          'Adanya penghitungan suara ulang di Pemilu/Pilkada.',
          'Adanya Pemungutan suara susulan di Pemilu/Pilkada.',
          'Adanya saran perbaikan (catatan khusus) dari pengawas saat pemungutan suara.',
          'Adanya komplain dari saksi saat pemungutan atau penghitungan suara.',
          'Adanya gugatan atas hasil Pemilu/Pilkada.',
          'Adanya keberatan dan/atau sengketa proses Pemilu/Pilkada.',
          'Adanya TPS yang tidak dihadiri oleh Pengawas TPS pada Pemilu/Pilkada.',
          'Adanya Informasi ketiadaan Pengawas Pemilu Tingkat Kelurahan/Desa.',
          'Adanya Penolakan terhadap calon perempuan di Pemilu/Pilkada.',
          'Adanya Penolakan terhadap calon berlatarbelakang etnis, Suku, dan Agama tertentu di Pemilu/Pilkada.',
          'Adanya dokumen palsu dalam proses pencalonan di Pemilu dan Pilkada.',
          'Adanya informasi kampanye di luar jadwal yang dilakukan oleh peserta Pemilu.',
          'Adanya iklan kampanye di luar jadwal yang dilakukan oleh peserta Pemilu.',
          'Adanya materi kampanye yang bermuatan SARA di tempat umum.',
          'Adanya materi kampanye yang bermuata SARA di sosial media akun lokal (Grup WA / Facebook).',
          'Adanya materi kampanye Hoax di tempat umum.',
          'Adanya materi kampanye Hoax di sosial media (Grup WA / Facebook).',
          'Adanya materi kampanye ujaran kebencian di tempat umum.',
          'Adanya materi kampanye ujaran kebencian di sosial media (Grup WA / Facebook).',
          'Adanya pelanggaran lokasi kampanye yang dilakukan oleh peserta Pemilu/Pilkada.',
          'Adanya konflik antar pendukung peserta pemilu/pasangan calon.',
          'Adanya Laporan tentang Politik Uang yang dilakukan oleh Peserta/Tim Sukses/Tim Kampanye Pemilu.',
          'Adanya peserta pemilu/calon yang tidak melaporkan dana kampanye.',
          'Adanya upaya untuk menghalanghalangi Pemilih memberikan suara di TPS.',
          'Adanya mobilisasi pemilih tambahan secara mendadak.',
          'Adanya pemilih tambahan melebihi jumlah surat suara cadangan 2%.',
          'Adanya laporan masyarakat yang tidak ditindaklanjuti oleh Pengawas Pemilu.',
          'Adanya Informasi tentang ketiadaan pemantau Pemilu.',
          'Adanya mobilisasi penolakan terhadap pelaksanaan pemilu/pilkada dari kelompok masyarakat.'
]

# ========================
# 2. Tokenisasi
# ========================

corpus_tokenized = [simple_preprocess(doc) for doc in corpus_data]
query_tokenized = [simple_preprocess(query) for query in query_data]

# ========================
# 3. Latih model Word2Vec
# ========================

model = Word2Vec(sentences=query_tokenized + corpus_tokenized,
                 vector_size=100, window=5, min_count=1, workers=4, sg=1, epochs=100 )

# ========================
# 4. Fungsi bantu
# ========================

def sentence_vector(sentence, model):
    words = [word for word in sentence if word in model.wv]
    if len(words) == 0:
        return np.zeros(model.vector_size)
    return np.mean([model.wv[word] for word in words], axis=0)

# Hitung vektor untuk semua dokumen dalam korpus
corpus_vectors = [sentence_vector(doc, model) for doc in corpus_tokenized]

# ========================
# 5. Hitung kemiripan
# ========================

results = []
for q_idx, query in enumerate(query_tokenized):
    query_vec = sentence_vector(query, model)
    similarities = [
        (doc_idx, cosine_similarity([query_vec], [doc_vec])[0][0])
        for doc_idx, doc_vec in enumerate(corpus_vectors)
    ]
    similarities.sort(key=lambda x: x[1], reverse=True)

    # Dokumen paling mirip
    most_similar = similarities[0]
    results.append({
        "Query_Index": q_idx + 1,
        "Query": ' '.join(query),
        "Top_Doc_Index": most_similar[0] + 1,
        "Top_Document": corpus_data[most_similar[0]],
        "Similarity": most_similar[1],
        "Type": "paling mirip"
    })

# ========================
# 6. Simpan ke Excel
# ========================

df_result = pd.DataFrame(results)
df_result.to_excel("hasil_kemiripan_word2vec_ASLII.xlsx", index=False)

print("Proses selesai. Hasil disimpan di 'hasil_kemiripan_word2vec.xlsx'.")


Proses selesai. Hasil disimpan di 'hasil_kemiripan_word2vec.xlsx'.


In [5]:
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

# ========================
# 1. Load data
# ========================

# Query diambil dari Excel
query_dataframe = pd.read_excel('step6_final.xlsx')  # Ganti nama file jika perlu
query_data = query_dataframe['hasil_preprocessing'].dropna().tolist()  # Pastikan tidak ada NaN

# Korpus adalah daftar isu pemilu (dimasukkan sebagai list Python)
corpus_data = [
    'Adanya kekerasan/kerusuhan berbasis SARA dalam isu-isu politik dan kepemiluan.',
          'Adanya kekerasan/kerusuhan yang melibatkan tokoh publik/politik/aparat keamanan.',
          'Adanya bencana alam yang mengganggu tahapan Pemilu/Pilkada.',
          'Adanya bencana non alam (pandemi covid 19) yang mengganggu tahapan Pemilu/Pilkada.',
          'Adanya Intimidasi Terhadap Peserta Pemilu dalam proses pelaksanaan Pemilu/Pilkada.',
          'Adanya Intimidasi Terhadap Penyelenggara Pemilu dalam proses pelaksanaan Pemilu/Pilkada.',
          'Adanya Intimidasi Terhadap Pemilih dalam proses pelaksanaan Pemilu/Pilkada.',
          'Adanya perusakan fasilitas penyelenggaraan Pemilu.',
          'Adanya putusan DKPP yang ditujukan kepada Jajaran KPU dan/atau Bawaslu.',
          'Adanya Rekomendasi dan/atau Putusan Bawaslu yang tidak ditindaklanjuti oleh KPU.',
          'Adanya rekomendasi Bawaslu terkait dengan perubahan suara pada proses rekapitulasi suara.',
          'Adanya Rekomendasi Bawaslu terkait ketidaknetralan ASN/TNI/POLRI.',
          'Adanya penyalahgunaan anggaran Pemilu/Pilkada.',
          'Adanya intimidasi kepada penyelenggara pemilu yang dilakukan oleh Kepala Daerah.',
          'Adanya himbauan dan/atau tindakan untuk memilih calon tertentu dari pemerintah lokal.',
          'Adanya himbauan dan/atau tindakan untuk menolak calon tertentu dari pemerintah lokal atau tokoh masyarakat.',
          'Adanya pemilih memenuhi syarat tetapi tidak terdaftar dalam daftar pemilih tetap.',
          'Adanya pemilih tidak memenuhi syarat terdaftar dalam pemilih tetap.',
          'Adanya penduduk potensial memilih tetapi tidak memiliki KTP-Elektronik.',
          'Adanya pemilih ganda dalam daftar pemilih (logistik).',
          'Adanya penggelembungan jumlah data pemilih.',
          'Adanya Pemilih Pindah Memilih (DPTB) yang Tidak Dapat Memberikan Hak Suaranya.',
          'Adanya ketidakprofesionalan Penyelenggara Pemilu yang merugikan kampanye calon.',
          'Adanya kampanye yang difasilitasi KPU tidak sesuai dengan ketentuan.',
          'Adanya penyelenggara Pemilu yang menunjukan sikap keberpihakan dalam tahapan kampanye.',
          'Adanya penyelenggara Pemilu yang melakukan pengabaian terhadap protokol kesehatan dalam kegiatan kampanye.',
          'Adanya perlengkapan pemungutan suara yang tidak sesuai dengan ketentuan Ada keterlambatan perlengkapan (logistik) pemungutan suara.',
          'Ada keterlambatan perlengkapan (logistik) pemungutan suara.',
          'Adanya logistik berupa surat suara pemungutan suara yang tertukar.',
          'Adanya informasi tidak terdapat TPS untuk menfasilitasi pemungutan suara bagi pemilih di LAPAS/RS/SEKOLAH DINAS/.',
          'Adanya informasi pelanggaraan saat pemungutan suara di Pemilu/Pilkada.',
          'Adanya pemungutan suara ulang di Pemilu/Pilkada.',
          'Adanya penghitungan suara ulang di Pemilu/Pilkada.',
          'Adanya Pemungutan suara susulan di Pemilu/Pilkada.',
          'Adanya saran perbaikan (catatan khusus) dari pengawas saat pemungutan suara.',
          'Adanya komplain dari saksi saat pemungutan atau penghitungan suara.',
          'Adanya gugatan atas hasil Pemilu/Pilkada.',
          'Adanya keberatan dan/atau sengketa proses Pemilu/Pilkada.',
          'Adanya TPS yang tidak dihadiri oleh Pengawas TPS pada Pemilu/Pilkada.',
          'Adanya Informasi ketiadaan Pengawas Pemilu Tingkat Kelurahan/Desa.',
          'Adanya Penolakan terhadap calon perempuan di Pemilu/Pilkada.',
          'Adanya Penolakan terhadap calon berlatarbelakang etnis, Suku, dan Agama tertentu di Pemilu/Pilkada.',
          'Adanya dokumen palsu dalam proses pencalonan di Pemilu dan Pilkada.',
          'Adanya informasi kampanye di luar jadwal yang dilakukan oleh peserta Pemilu.',
          'Adanya iklan kampanye di luar jadwal yang dilakukan oleh peserta Pemilu.',
          'Adanya materi kampanye yang bermuatan SARA di tempat umum.',
          'Adanya materi kampanye yang bermuata SARA di sosial media akun lokal (Grup WA / Facebook).',
          'Adanya materi kampanye Hoax di tempat umum.',
          'Adanya materi kampanye Hoax di sosial media (Grup WA / Facebook).',
          'Adanya materi kampanye ujaran kebencian di tempat umum.',
          'Adanya materi kampanye ujaran kebencian di sosial media (Grup WA / Facebook).',
          'Adanya pelanggaran lokasi kampanye yang dilakukan oleh peserta Pemilu/Pilkada.',
          'Adanya konflik antar pendukung peserta pemilu/pasangan calon.',
          'Adanya Laporan tentang Politik Uang yang dilakukan oleh Peserta/Tim Sukses/Tim Kampanye Pemilu.',
          'Adanya peserta pemilu/calon yang tidak melaporkan dana kampanye.',
          'Adanya upaya untuk menghalanghalangi Pemilih memberikan suara di TPS.',
          'Adanya mobilisasi pemilih tambahan secara mendadak.',
          'Adanya pemilih tambahan melebihi jumlah surat suara cadangan 2%.',
          'Adanya laporan masyarakat yang tidak ditindaklanjuti oleh Pengawas Pemilu.',
          'Adanya Informasi tentang ketiadaan pemantau Pemilu.',
          'Adanya mobilisasi penolakan terhadap pelaksanaan pemilu/pilkada dari kelompok masyarakat.'
]

# ========================
# 2. Tokenisasi
# ========================

corpus_tokenized = [simple_preprocess(doc) for doc in corpus_data]
query_tokenized = [simple_preprocess(query) for query in query_data]

# ========================
# 3. Latih model Word2Vec
# ========================

model = Word2Vec(sentences=query_tokenized + corpus_tokenized,
                 vector_size=100, window=5, min_count=1, workers=4, sg=1, epochs=100 )

# ========================
# 4. Fungsi bantu
# ========================

def sentence_vector(sentence, model):
    words = [word for word in sentence if word in model.wv]
    if len(words) == 0:
        return np.zeros(model.vector_size)
    return np.mean([model.wv[word] for word in words], axis=0)

# Hitung vektor untuk semua dokumen dalam korpus
corpus_vectors = [sentence_vector(doc, model) for doc in corpus_tokenized]

# ========================
# 5. Hitung kemiripan
# ========================

results = []
for q_idx, query in enumerate(query_tokenized):
    query_vec = sentence_vector(query, model)
    similarities = [
        (doc_idx, cosine_similarity([query_vec], [doc_vec])[0][0])
        for doc_idx, doc_vec in enumerate(corpus_vectors)
    ]
    similarities.sort(key=lambda x: x[1], reverse=True)

    # Dokumen paling mirip
    most_similar = similarities[0]
    results.append({
        "Query_Index": q_idx + 1,
        "Query": ' '.join(query),
        "Top_Doc_Index": most_similar[0] + 1,
        "Top_Document": corpus_data[most_similar[0]],
        "Similarity": most_similar[1],
        "Type": "paling mirip"
    })

    # Dokumen paling tidak mirip
    least_similar = similarities[-1]
    results.append({
        "Query_Index": q_idx + 1,
        "Query": ' '.join(query),
        "Top_Doc_Index": least_similar[0] + 1,
        "Top_Document": corpus_data[least_similar[0]],
        "Similarity": least_similar[1],
        "Type": "paling tidak mirip"
    })

# ========================
# 6. Simpan ke Excel
# ========================

df_result = pd.DataFrame(results)
df_result.to_excel("hasil_kemiripan_word2vec_miripDANtidakmirippp.xlsx", index=False)

print("Proses selesai. Hasil disimpan di 'hasil_kemiripan_word2vec.xlsx'.")


Proses selesai. Hasil disimpan di 'hasil_kemiripan_word2vec.xlsx'.
